# The Complete Guide to Polars for Data Science

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/norochalise/noroinsight-polars-tutorial/blob/main/Complete_Guide_to_Polars.ipynb)

This is part of the original article [The Complete Guide to Polars for Data Science](https://noroinsight.com/polars-for-data-science-complete-guide/)

## Dataset & Code Repository

This guide uses the Customer Transactions & Spending Dataset (1M Rows), which includes customer demographics, spending behavior, and payment details:

- [Kaggle Dataset: Customer Transactions & Spending (1M Rows)](https://www.kaggle.com/datasets/mountboy/online-store-customer-transactions-1m-rows)
- [Read the full article on Noro Insight](https://noroinsight.com/polars-for-data-science-complete-guide/)

Every cell is tested on **Polars 2.0** with the current version of the dataset. The notebook downloads the data for you.

## 1. Setup

In [1]:
# Install Polars and kagglehub (to download the dataset)
!pip install -q "polars>=2.0" kagglehub

In [2]:
# Import libraries
import os
import polars as pl
import kagglehub

print("Polars", pl.__version__)  # This notebook is tested on Polars 2.0

Polars 2.0.0


**Download the dataset**

This downloads the dataset from Kaggle (no login needed). If you already have the CSV, set `DATA_PATH` to its location instead.

In [3]:
DATA_PATH = os.path.join(
    kagglehub.dataset_download("mountboy/online-store-customer-transactions-1m-rows"),
    "customer_spending_1M_2018_2025.csv",
)
DATA_PATH

In [4]:
# Setting Display Options
import polars as pl

pl.Config.set_tbl_rows(10)  # Show 10 rows
pl.Config.set_tbl_cols(15)   # Show 15 columns
pl.Config.set_fmt_str_lengths(90)  # Increase max string column width

polars.config.Config

## 2. Loading and Saving Data

**Reading CSV Files**

In [5]:
# Reading CSV Files
df = pl.read_csv(DATA_PATH)
df.head()

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",1,153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",1,293.58
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01


**Reading Other File Formats**

In [6]:
# Create small example files in other formats so this cell runs on its own
df.head(1000).write_parquet("online_store_data.parquet")
df.head(1000).write_json("customer_data.json")
df.head(1000).write_ipc("online_store_data.feather")

# Read a Parquet file (Recommended for large datasets)
df_parquet = pl.read_parquet("online_store_data.parquet")

# Read a JSON file
df_json = pl.read_json("customer_data.json")

# Read an IPC/Feather file (Optimized for fast in-memory processing)
df_ipc = pl.read_ipc("online_store_data.feather")

# Read an Avro file (Commonly used for efficient data pipelines)
# df_avro = pl.read_avro("online_store_data.avro")

**Reading CSV with Additional Parameters**

In [7]:
df_skipped = pl.read_csv(
    DATA_PATH,
    skip_rows=2  # Skip first two rows if they contain metadata
)

**Handling Files Without a Header Row**

By default the first row becomes the column names. For a file with no header row, say so and supply the names yourself:

In [8]:
# Create a small example file with no header row (4 columns)
df.head(5).select("Transaction_ID", "Transaction_date", "Gender", "Age").write_csv("customer_data_no_header.csv", include_header=False)

df_no_header = pl.read_csv(
    "customer_data_no_header.csv",
    has_header=False,  # The first row is data, not column names
    new_columns=["Transaction_ID", "Transaction_date", "Gender", "Age"],
)
df_no_header

Transaction_ID,Transaction_date,Gender,Age
i64,str,str,i64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39
1001,"""2018-01-01T00:06:00.000000""","""Male""",34
1002,"""2018-01-01T00:14:00.000000""","""Female""",53
1003,"""2018-01-01T00:23:00.000000""","""Male""",33
1004,"""2018-01-01T00:25:00.000000""","""Female""",36


**Reading CSV from a URL**

In [9]:
df_url = pl.read_csv("https://raw.githubusercontent.com/norochalise/pandas-tutorial-article-2022/refs/heads/main/dataset/online_store_customer_data.csv")

**Streaming Large Datasets with scan_csv()**

In [10]:
df_stream = pl.scan_csv(DATA_PATH)

# Apply filtering before loading into memory
df_filtered = df_stream.filter(pl.col("Amount_spent") > 50)

# Collect to load the required data into memory
df_final = df_filtered.collect()
df_final.head(3)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",1,153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96


**Saving Data Files**

In [11]:
# Save DataFrame as CSV
df.write_csv("customer_data.csv")

# Save DataFrame as Parquet (recommended for large datasets)
df.write_parquet("customer_data.parquet")

# Save DataFrame as JSON
df.write_json("customer_data.json")

# Save DataFrame as IPC/Feather
df.write_ipc("customer_data.feather")


## 3. Data Exploration and Manipulation

### Viewing Data (Head, Tail, Schema, Describe)

**Checking the First Few Rows**

In [12]:
df.head()  # Shows the first 5 rows

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",1,153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",1,293.58
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01


**Viewing the Last Few Rows**

In [13]:
df.tail()  # Shows the last 5 rows

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000995,"""2025-01-30T23:43:00.000000""","""Male""",45,"""Single""","""Arkansas""","""Gold""","""Employees""","""PayPal""",1,1762.76
1000996,"""2025-01-30T23:45:00.000000""","""Female""",75,"""Married""","""Virginia""","""Silver""","""workers""","""Other""",1,31.41
1000997,"""2025-01-30T23:47:00.000000""","""Female""",61,"""Married""","""Kansas""","""Basic""","""self-employed""","""Other""",1,1699.31
1000998,"""2025-01-30T23:52:00.000000""","""Female""",68,"""Married""","""Utah""","""Basic""","""Employees""","""PayPal""",1,208.26
1000999,"""2025-01-30T23:57:00.000000""","""Male""",55,"""Single""","""Nevada""","""Missing""","""workers""","""PayPal""",0,2063.92


**Retrieving Sample Rows from a DataFrame**

In [14]:
df.sample(n=4)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
196474,"""2019-05-23T01:08:00.000000""","""Male""",34,"""Married""","""South Dakota""","""Basic""",null,"""PayPal""",1,786.73
712411,"""2023-01-16T10:24:00.000000""","""Male""",49,"""Married""","""Delaware""","""Platinum""","""Employees""","""Other""",0,2454.37
532148,"""2021-10-06T00:14:00.000000""","""Female""",76,"""Single""","""Kansas""","""Silver""","""Employees""","""PayPal""",1,82.27
153532,"""2019-01-31T19:02:00.000000""","""Male""",51,"""Married""","""Alabama""","""Platinum""","""self-employed""","""Other""",1,1367.77


**Checking the Structure of the Data (df.schema)**

In [15]:
df.schema

Schema([('Transaction_ID', Int64),
        ('Transaction_date', String),
        ('Gender', String),
        ('Age', Int64),
        ('Marital_status', String),
        ('State_names', String),
        ('Segment', String),
        ('Employees_status', String),
        ('Payment_method', String),
        ('Referral', Int64),
        ('Amount_spent', Float64)])

**Summarizing Data**

In [16]:
df.describe()

statistic,Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
str,f64,str,str,f64,str,str,str,str,str,f64,f64
"""count""",1e6,"""1000000""","""988948""",983346.0,"""1000000""","""1000000""","""1000000""","""989775""","""1000000""",938188.0,903935.0
"""null_count""",0.0,"""0""","""11052""",16654.0,"""0""","""0""","""0""","""10225""","""0""",61812.0,96065.0
"""mean""",500999.5,null,null,46.64717,null,null,null,null,null,0.652339,1416.126106
"""std""",288675.278932,null,null,18.183986,null,null,null,null,null,0.476228,878.123082
"""min""",1000.0,"""2018-01-01T00:04:00.000000""","""Female""",15.0,"""Married""","""Alabama""","""Basic""","""Employees""","""Card""",0.0,2.09
"""25%""",251000.0,null,null,32.0,null,null,null,null,null,0.0,677.36
"""50%""",501000.0,null,null,47.0,null,null,null,null,null,1.0,1332.48
"""75%""",750999.0,null,null,62.0,null,null,null,null,null,1.0,2037.59
"""max""",1.000999e6,"""2025-01-30T23:57:00.000000""","""Male""",78.0,"""Single""","""Wyoming""","""Silver""","""workers""","""PayPal""",1.0,2999.98


### Selecting, Filtering, and Slicing Data

**Selecting Specific Columns**

In [17]:
#Selecting One Column
df.select("Amount_spent")

Amount_spent
f64
1557.5
153.55
2640.96
293.58
1608.01
…
1762.76
31.41
1699.31


In [18]:
# Selecting Multiple Columns
df.select(["Amount_spent", "Transaction_ID", "Gender"])

Amount_spent,Transaction_ID,Gender
f64,i64,str
1557.5,1000,"""Female"""
153.55,1001,"""Male"""
2640.96,1002,"""Female"""
293.58,1003,"""Male"""
1608.01,1004,"""Female"""
…,…,…
1762.76,1000995,"""Male"""
31.41,1000996,"""Female"""
1699.31,1000997,"""Female"""


#### Filtering Rows Based on Conditions

In [19]:
# Filtering Rows Based on a Single Condition
df.filter(pl.col("Amount_spent") > 50)


Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",1,153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",1,293.58
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01
…,…,…,…,…,…,…,…,…,…,…
1000994,"""2025-01-30T23:29:00.000000""","""Female""",51,"""Single""","""West Virginia""","""Platinum""","""Unemployment""","""Card""",0,522.04
1000995,"""2025-01-30T23:43:00.000000""","""Male""",45,"""Single""","""Arkansas""","""Gold""","""Employees""","""PayPal""",1,1762.76
1000997,"""2025-01-30T23:47:00.000000""","""Female""",61,"""Married""","""Kansas""","""Basic""","""self-employed""","""Other""",1,1699.31


In [20]:
# Filtering Rows Based on Multiple Conditions
df.filter((pl.col("Amount_spent") > 100) & (pl.col("Gender") == "Female"))

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01
1006,"""2018-01-01T00:29:00.000000""","""Female""",55,"""Single""","""Ohio""","""Silver""","""Unemployment""","""PayPal""",0,1553.61
1007,"""2018-01-01T00:29:00.000000""","""Female""",35,"""Married""","""Hawaii""","""Basic""","""workers""","""Other""",1,1851.58
…,…,…,…,…,…,…,…,…,…,…
1000992,"""2025-01-30T23:23:00.000000""","""Female""",25,"""Married""","""Nebraska""","""Basic""","""workers""","""PayPal""",0,1735.07
1000993,"""2025-01-30T23:29:00.000000""","""Female""",18,"""Married""","""Alabama""","""Silver""","""workers""","""PayPal""",null,2091.73
1000994,"""2025-01-30T23:29:00.000000""","""Female""",51,"""Single""","""West Virginia""","""Platinum""","""Unemployment""","""Card""",0,522.04


In [21]:
df.filter((pl.col("Marital_status") == "Married") & (pl.col("State_names") == "California"))

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1085,"""2018-01-01T04:59:00.000000""","""Male""",33,"""Married""","""California""","""Basic""","""Employees""","""Other""",0,2621.91
1107,"""2018-01-01T06:11:00.000000""","""Male""",65,"""Married""","""California""","""Basic""","""Unemployment""","""PayPal""",1,884.14
1140,"""2018-01-01T08:41:00.000000""","""Female""",60,"""Married""","""California""","""Basic""","""Unemployment""","""Other""",0,2785.78
1335,"""2018-01-01T21:20:00.000000""","""Male""",61,"""Married""","""California""","""Basic""","""self-employed""","""PayPal""",1,670.59
1395,"""2018-01-02T01:42:00.000000""","""Female""",28,"""Married""","""California""","""Gold""","""workers""","""Card""",1,1887.3
…,…,…,…,…,…,…,…,…,…,…
1000583,"""2025-01-30T00:04:00.000000""","""Female""",22,"""Married""","""California""","""Basic""","""workers""","""PayPal""",1,1510.28
1000586,"""2025-01-30T00:12:00.000000""","""Male""",49,"""Married""","""California""","""Basic""","""workers""","""PayPal""",1,337.17
1000632,"""2025-01-30T02:26:00.000000""","""Female""",28,"""Married""","""California""","""Basic""","""Employees""","""PayPal""",1,1287.31


### Extracting a Specific Range of Rows (Slicing)

**Extract a Range of Rows by Index**

In [22]:
df.slice(10, 10)  # Start at index 10, retrieve 10 rows (rows 10 to 19)


Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1010,"""2018-01-01T00:34:00.000000""","""Female""",40,"""Single""","""Illinois""","""Basic""","""Unemployment""","""PayPal""",1,1160.65
1011,"""2018-01-01T00:36:00.000000""","""Male""",73,"""Married""","""West Virginia""","""Basic""","""Employees""","""PayPal""",0,1397.09
1012,"""2018-01-01T00:41:00.000000""","""Female""",44,"""Single""","""Florida""","""Basic""","""workers""","""PayPal""",0,1778.33
1013,"""2018-01-01T00:43:00.000000""","""Male""",77,"""Married""","""Oregon""","""Missing""","""workers""","""Card""",0,1449.98
1014,"""2018-01-01T00:43:00.000000""","""Male""",54,"""Single""","""North Carolina""","""Platinum""","""Employees""","""Card""",1,2936.96
1015,"""2018-01-01T00:44:00.000000""","""Female""",37,"""Married""","""Arizona""","""Basic""","""Employees""","""Other""",1,null
1016,"""2018-01-01T00:52:00.000000""","""Female""",75,"""Single""","""Tennessee""","""Platinum""","""Employees""","""Card""",0,2870.69
1017,"""2018-01-01T00:53:00.000000""","""Female""",77,"""Single""","""Washington""","""Platinum""","""Employees""","""PayPal""",0,439.56
1018,"""2018-01-01T00:57:00.000000""","""Male""",55,"""Married""","""South Carolina""","""Basic""","""Employees""","""PayPal""",1,1274.62


In [23]:
df.slice(0, 100)  # Retrieve the first 100 rows

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",1,153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",1,293.58
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01
…,…,…,…,…,…,…,…,…,…,…
1095,"""2018-01-01T05:44:00.000000""","""Male""",24,"""Single""","""Washington""","""Platinum""","""workers""","""PayPal""",0,2042.64
1096,"""2018-01-01T05:45:00.000000""","""Male""",44,"""Married""","""Indiana""","""Basic""","""self-employed""","""PayPal""",1,null
1097,"""2018-01-01T05:47:00.000000""","""Female""",20,"""Married""","""Nebraska""","""Missing""","""Employees""","""PayPal""",1,1324.12


In [24]:
df.tail(50)  # Get the last 50 rows

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000950,"""2025-01-30T20:27:00.000000""","""Male""",72,"""Married""","""Minnesota""","""Silver""","""Employees""","""PayPal""",0,892.56
1000951,"""2025-01-30T20:30:00.000000""","""Male""",78,"""Married""","""New Mexico""","""Basic""","""Unemployment""","""Card""",1,95.05
1000952,"""2025-01-30T20:31:00.000000""","""Male""",19,"""Single""","""Vermont""","""Basic""","""self-employed""","""PayPal""",1,1967.6
1000953,"""2025-01-30T20:33:00.000000""","""Female""",63,"""Single""","""Louisiana""","""Basic""","""Employees""","""Other""",1,958.83
1000954,"""2025-01-30T20:34:00.000000""","""Male""",70,"""Single""","""New Jersey""","""Basic""","""workers""","""PayPal""",0,2969.24
…,…,…,…,…,…,…,…,…,…,…
1000995,"""2025-01-30T23:43:00.000000""","""Male""",45,"""Single""","""Arkansas""","""Gold""","""Employees""","""PayPal""",1,1762.76
1000996,"""2025-01-30T23:45:00.000000""","""Female""",75,"""Married""","""Virginia""","""Silver""","""workers""","""Other""",1,31.41
1000997,"""2025-01-30T23:47:00.000000""","""Female""",61,"""Married""","""Kansas""","""Basic""","""self-employed""","""Other""",1,1699.31


### Sorting and Applying Conditions

**Sorting Data in Ascending Order**

In [25]:
df.sort("Amount_spent", nulls_last=True).head(4)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
9032,"""2018-01-21T18:02:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09
12093,"""2018-01-29T13:52:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09
14260,"""2018-02-04T03:34:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09
16267,"""2018-02-09T09:43:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09


**Sorting Data in Descending Order**

In [26]:
df.sort("Amount_spent", descending=True)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1008,"""2018-01-01T00:31:00.000000""","""Male""",18,"""Married""","""Nevada""","""Basic""","""Employees""","""Card""",1,null
1015,"""2018-01-01T00:44:00.000000""","""Female""",37,"""Married""","""Arizona""","""Basic""","""Employees""","""Other""",1,null
1025,"""2018-01-01T01:13:00.000000""","""Male""",21,"""Married""","""Florida""","""Silver""","""Unemployment""","""Card""",0,null
1039,"""2018-01-01T01:51:00.000000""","""Female""",37,"""Single""","""Washington""","""Basic""","""self-employed""","""PayPal""",1,null
1063,"""2018-01-01T03:41:00.000000""","""Male""",39,"""Married""","""Montana""","""Platinum""","""Unemployment""","""Other""",0,null
…,…,…,…,…,…,…,…,…,…,…
989972,"""2025-01-02T13:57:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09
989980,"""2025-01-02T14:28:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09
993301,"""2025-01-11T12:05:00.000000""","""Female""",73,"""Married""","""Michigan""","""Gold""","""Employees""","""PayPal""",1,2.09


**Sorting by Multiple Columns**

In [27]:
df.sort(["Gender", "Amount_spent"], descending=[False, True], nulls_last=True).head(4)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1270,"""2018-01-01T16:21:00.000000""","""Female""",77,"""Married""","""New Mexico""","""Platinum""","""workers""","""Card""",0,2999.98
3182,"""2018-01-06T20:42:00.000000""","""Female""",77,"""Married""","""New Mexico""","""Platinum""","""workers""","""Card""",0,2999.98
7065,"""2018-01-16T18:30:00.000000""","""Female""",77,"""Married""","""New Mexico""","""Platinum""","""workers""","""Card""",0,2999.98
8747,"""2018-01-21T00:15:00.000000""","""Female""",77,"""Married""","""New Mexico""","""Platinum""","""workers""","""Card""",0,2999.98


### Applying Complex Conditional Queries

**Find Female Customers Who Spent More Than $100**

In [28]:
df.filter((pl.col("Amount_spent") > 100) & (pl.col("Gender") == "Female"))

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",1,2640.96
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01
1006,"""2018-01-01T00:29:00.000000""","""Female""",55,"""Single""","""Ohio""","""Silver""","""Unemployment""","""PayPal""",0,1553.61
1007,"""2018-01-01T00:29:00.000000""","""Female""",35,"""Married""","""Hawaii""","""Basic""","""workers""","""Other""",1,1851.58
…,…,…,…,…,…,…,…,…,…,…
1000992,"""2025-01-30T23:23:00.000000""","""Female""",25,"""Married""","""Nebraska""","""Basic""","""workers""","""PayPal""",0,1735.07
1000993,"""2025-01-30T23:29:00.000000""","""Female""",18,"""Married""","""Alabama""","""Silver""","""workers""","""PayPal""",null,2091.73
1000994,"""2025-01-30T23:29:00.000000""","""Female""",51,"""Single""","""West Virginia""","""Platinum""","""Unemployment""","""Card""",0,522.04


**Get Customers Aged Between 25 and 40**

In [29]:
df.filter((pl.col("Age") >= 25) & (pl.col("Age") <= 40))

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
i64,str,str,i64,str,str,str,str,str,i64,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",0,1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",1,153.55
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",1,293.58
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",0,1608.01
1007,"""2018-01-01T00:29:00.000000""","""Female""",35,"""Married""","""Hawaii""","""Basic""","""workers""","""Other""",1,1851.58
…,…,…,…,…,…,…,…,…,…,…
1000978,"""2025-01-30T22:33:00.000000""","""Female""",31,"""Single""","""New York""","""Silver""","""self-employed""","""PayPal""",1,null
1000979,"""2025-01-30T22:36:00.000000""","""Female""",40,"""Single""","""Alaska""","""Platinum""","""workers""","""Other""",1,null
1000985,"""2025-01-30T23:11:00.000000""","""Female""",37,"""Married""","""Nevada""","""Silver""","""self-employed""","""Other""",0,507.25


## 4. Data Cleaning and Preprocessing

### Handling Missing Values

**Checking for Missing Values**

In [30]:
df_clean = df
df_clean.null_count()

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,11052,16654,0,0,0,10225,0,61812,96065


**Dropping Missing Values**

In [31]:
df_clean = df.drop("Referral")  # Removes the 'Referral' column
df_clean.head(4)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Amount_spent
i64,str,str,i64,str,str,str,str,str,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",2640.96
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",293.58


In [32]:
df_clean.drop_nulls(subset=["Gender"])

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Amount_spent
i64,str,str,i64,str,str,str,str,str,f64
1000,"""2018-01-01T00:04:00.000000""","""Female""",39,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""",1557.5
1001,"""2018-01-01T00:06:00.000000""","""Male""",34,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""",153.55
1002,"""2018-01-01T00:14:00.000000""","""Female""",53,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""",2640.96
1003,"""2018-01-01T00:23:00.000000""","""Male""",33,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""",293.58
1004,"""2018-01-01T00:25:00.000000""","""Female""",36,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""",1608.01
…,…,…,…,…,…,…,…,…,…
1000995,"""2025-01-30T23:43:00.000000""","""Male""",45,"""Single""","""Arkansas""","""Gold""","""Employees""","""PayPal""",1762.76
1000996,"""2025-01-30T23:45:00.000000""","""Female""",75,"""Married""","""Virginia""","""Silver""","""workers""","""Other""",31.41
1000997,"""2025-01-30T23:47:00.000000""","""Female""",61,"""Married""","""Kansas""","""Basic""","""self-employed""","""Other""",1699.31


**Filling Missing Values (Imputation)**

In [33]:
# replace missing values in Gender with "Unknown":
df_clean = df_clean.with_columns(pl.col("Gender").fill_null("Unknown"))


In [34]:
# replace missing values with zero or another fixed value:
df_clean = df_clean.with_columns(pl.col("Amount_spent").fill_null(0))

#### Filling with Statistical Measures

**Replace missing values in Age with the median:**

In [35]:
median_age = df_clean["Age"].median()
df_clean = df_clean.with_columns(pl.col("Age").fill_null(median_age))

**Replace missing values in Amount_spent with the mean:**

In [36]:
df_clean = df
mean_spent = df_clean["Amount_spent"].mean()
df_clean = df_clean.with_columns(pl.col("Amount_spent").fill_null(mean_spent))

**Replace missing values in Gender with the most frequent value (mode):**

In [37]:
mode_gender = df["Gender"].mode()[0]  # Get most common value
df = df.with_columns(pl.col("Gender").fill_null(mode_gender))

**Forward Filling** use the previous row’s value to fill the missing value

In [38]:
df_clean = df
df_clean = df_clean.with_columns(pl.col("Amount_spent").fill_null(strategy="forward"))

**Backward fill** (use the next row’s value to fill the missing value):

In [39]:
df_clean = df
df = df.with_columns(pl.col("Amount_spent").fill_null(strategy="backward"))

### Data Type Conversion

**Checking Data Types**

In [40]:
df.schema

Schema([('Transaction_ID', Int64),
        ('Transaction_date', String),
        ('Gender', String),
        ('Age', Int64),
        ('Marital_status', String),
        ('State_names', String),
        ('Segment', String),
        ('Employees_status', String),
        ('Payment_method', String),
        ('Referral', Int64),
        ('Amount_spent', Float64)])

**Convert Referral from Int64 to String**

In [41]:
df = df.with_columns(pl.col("Referral").cast(pl.Utf8))  # Convert to String for better readability

In [42]:
df.schema

Schema([('Transaction_ID', Int64),
        ('Transaction_date', String),
        ('Gender', String),
        ('Age', Int64),
        ('Marital_status', String),
        ('State_names', String),
        ('Segment', String),
        ('Employees_status', String),
        ('Payment_method', String),
        ('Referral', String),
        ('Amount_spent', Float64)])

**Convert Age from Int64 to Float64**

In [43]:
df = df.with_columns(pl.col("Age").cast(pl.Float64))


**Convert `Transaction_date` from text to a real timestamp**

The CSV stores each transaction time as text, such as `2018-01-01T00:04:00.000000`. Parse it so date functions work, then format it any way you like.

In [44]:
df = df.with_columns(pl.col("Transaction_date").str.to_datetime())

# Once it is a datetime, format it for a report:
df = df.with_columns(pl.col("Transaction_date").dt.strftime("%d %b %Y").alias("Pretty_date"))  # 01 Jan 2018

In [45]:
df.head()

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent,Pretty_date
i64,datetime[μs],str,f64,str,str,str,str,str,str,f64,str
1000,2018-01-01 00:04:00,"""Female""",39.0,"""Single""","""Oklahoma""","""Platinum""","""Unemployment""","""Card""","""0""",1557.5,"""01 Jan 2018"""
1001,2018-01-01 00:06:00,"""Male""",34.0,"""Married""","""Hawaii""","""Basic""","""workers""","""PayPal""","""1""",153.55,"""01 Jan 2018"""
1002,2018-01-01 00:14:00,"""Female""",53.0,"""Married""","""Iowa""","""Basic""","""self-employed""","""PayPal""","""1""",2640.96,"""01 Jan 2018"""
1003,2018-01-01 00:23:00,"""Male""",33.0,"""Married""","""Wisconsin""","""Basic""","""self-employed""","""Card""","""1""",293.58,"""01 Jan 2018"""
1004,2018-01-01 00:25:00,"""Female""",36.0,"""Married""","""Texas""","""Platinum""","""Employees""","""Card""","""0""",1608.01,"""01 Jan 2018"""


## 5. Advanced Data Transformations

#### GroupBy and Aggregations

**To calculate total spending per gender:**

In [46]:
df.group_by("Gender").agg(pl.sum("Amount_spent"))

Gender,Amount_spent
str,f64
"""Female""",7.8495e8
"""Male""",6.3171e8


**To calculate total amount spent by Gender and Payment Method:**

In [47]:
df.group_by(["Gender", "Payment_method"]) \
  .agg(pl.sum("Amount_spent").cast(pl.Int64)) \
  .sort(["Gender", "Payment_method"])

Gender,Payment_method,Amount_spent
str,str,i64
"""Female""","""Card""",251092780
"""Female""","""Other""",186728178
"""Female""","""PayPal""",347128441
"""Male""","""Card""",166060495
"""Male""","""Other""",156921359
"""Male""","""PayPal""",308726924


**To calculate multiple statistics:**

In [48]:
df.group_by("Gender").agg([
    pl.sum("Amount_spent").alias("Total Spent"),
    pl.mean("Age").alias("Average Age"),
    pl.len().alias("Customer Count")
])

Gender,Total Spent,Average Age,Customer Count
str,f64,f64,u32
"""Male""",6.3171e8,46.494462,449117
"""Female""",7.8495e8,46.773035,550883


### Pivoting DataFrames

In [49]:
df.pivot(
    values="Amount_spent",
    index="Gender",
    on="Payment_method",
    aggregate_function="sum"
)

Gender,Card,PayPal,Other
str,f64,f64,f64
"""Female""",2.5109e8,3.4713e8,1.8673e8
"""Male""",1.6606e8,3.0873e8,1.5692e8


### Rolling Averages for Time Series

Total the spending per day, then average each day with the six before it (a 7-day rolling average). This uses the datetime column created in the data type section.

In [50]:
daily = (
    df.group_by(pl.col("Transaction_date").dt.date().alias("Day"))  # timestamp -> calendar day
    .agg(pl.sum("Amount_spent").alias("Daily_spent"))
    .sort("Day")
    .with_columns(pl.col("Daily_spent").rolling_mean(window_size=7).alias("7_day_avg"))
)
daily.tail(3)

Day,Daily_spent,7_day_avg
date,f64,f64
2025-01-28,530817.52,549678.815714
2025-01-29,540848.25,546990.124286
2025-01-30,551348.83,547740.904286


## 6. Performance Optimization Techniques

### Using LazyFrames for Efficient Execution

In [51]:
df_lazy = pl.scan_csv(DATA_PATH)

In [52]:
df_final = df_lazy.filter(pl.col("Amount_spent") > 100).collect()

**See what the optimizer does with `.explain()`**

The plan shows Polars reads only 2 of the 11 columns (`PROJECT 2/11 COLUMNS`) and filters while reading the file.

In [53]:
query = (
    pl.scan_csv(DATA_PATH)
    .filter(pl.col("Amount_spent") > 100)
    .group_by("Gender")
    .agg(pl.sum("Amount_spent"))
)
print(query.explain())

# For files bigger than your RAM, process in chunks with the streaming engine:
query.collect(engine="streaming")

Gender,Amount_spent
str,f64
null,1.3865e7
"""Female""",6.9032e8
"""Male""",5.7407e8


### Memory Management (Efficient Data Types)

**Converting Data Types for Optimization**

In [54]:
df = pl.read_csv(
    DATA_PATH,
    schema_overrides={"Transaction_ID": pl.Int32, "Amount_spent": pl.Float32}
)

**To check how much memory a DataFrame consumes:**

In [55]:
df.estimated_size(unit = 'mb')

86.55259037017822

### Multi-threaded Processing

**Parallel Processing in Aggregations**

In [56]:
df.group_by("Gender").agg(pl.sum("Amount_spent"))

Gender,Amount_spent
str,f32
null,1.3883243e7
"""Female""",6.91295296e8
"""Male""",5.74907328e8


In [57]:
# How many threads Polars uses (by default one per CPU core)
pl.thread_pool_size()

# To limit it, set this environment variable BEFORE importing polars (e.g. at the top of a script):
# import os; os.environ["POLARS_MAX_THREADS"] = "4"

16

## 7. SQL-like Queries with Polars

**Running SQL Queries in Polars**

In [58]:
# Quickest way: df.sql(), where the table is always called "self"
df.sql("SELECT Gender, SUM(Amount_spent) AS Total_spent FROM self GROUP BY Gender")

Gender,Total_spent
str,f32
null,1.3883243e7
"""Female""",6.91295296e8
"""Male""",5.74907328e8


In [59]:
# For queries across several tables, register them in a SQLContext
from polars import SQLContext

sql_context = SQLContext()

# Register DataFrame as a SQL table
sql_context.register("customers", df)

# Run SQL query (execute() returns a LazyFrame; collect() runs it)
result = sql_context.execute("SELECT Gender, SUM(Amount_spent) FROM customers GROUP BY Gender")
df_result = result.collect()
print(df_result)

shape: (3, 2)
┌────────┬──────────────┐
│ Gender ┆ Amount_spent │
│ ---    ┆ ---          │
│ str    ┆ f32          │
╞════════╪══════════════╡
│ null   ┆ 1.3883243e7  │
│ Female ┆ 6.91295296e8 │
│ Male   ┆ 5.74907328e8 │
└────────┴──────────────┘


**Filtering and Aggregation with SQL Queries**

In [60]:
result = sql_context.execute("""
    SELECT Gender, Payment_method, COUNT(*) AS Total_Transactions, AVG(Amount_spent) AS Avg_Spent
    FROM customers
    WHERE Amount_spent > 50
    GROUP BY Gender, Payment_method
    ORDER BY Avg_Spent DESC
""").collect()

In [61]:
print(result)

shape: (9, 4)
┌────────┬────────────────┬────────────────────┬─────────────┐
│ Gender ┆ Payment_method ┆ Total_Transactions ┆ Avg_Spent   │
│ ---    ┆ ---            ┆ ---                ┆ ---         │
│ str    ┆ str            ┆ i64                ┆ f32         │
╞════════╪════════════════╪════════════════════╪═════════════╡
│ Female ┆ Other          ┆ 113033             ┆ 1488.96936  │
│ Male   ┆ PayPal         ┆ 195568             ┆ 1462.975586 │
│ Female ┆ Card           ┆ 152513             ┆ 1449.810059 │
│ Male   ┆ Other          ┆ 99771              ┆ 1443.268555 │
│ Female ┆ PayPal         ┆ 211382             ┆ 1427.339722 │
│ null   ┆ Card           ┆ 4051               ┆ 1398.654053 │
│ null   ┆ PayPal         ┆ 4354               ┆ 1353.336182 │
│ Male   ┆ Card           ┆ 107065             ┆ 1350.705933 │
│ null   ┆ Other          ┆ 1852               ┆ 1255.329956 │
└────────┴────────────────┴────────────────────┴─────────────┘
